# 25 · Random Forest — El poder de preguntar a muchos árboles

**Pregunta:** si un árbol de decisión se equivoca cuando cambian los datos,
¿podemos construir algo más robusto?

Un Random Forest responde exactamente eso: en lugar de entrenar *un* árbol,
entrena **cientos** y les hace votar. Cada árbol ve una versión distinta de los datos
y solo puede examinar una parte de las variables en cada split.

En este notebook construimos ese proceso pieza por pieza,
desde el problema hasta un pipeline listo para producción.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

sns.set_theme(style="whitegrid", palette="muted")
rng = np.random.default_rng(42)

## 0 · El dataset — 400 estudiantes

Extendemos el dataset del notebook 24 con tres variables nuevas.

| Variable | Qué mide | Rango |
|---|---|---|
| `horas_estudio` | Horas de estudio el día anterior | 0–10 h |
| `horas_sueno` | Horas de sueño la noche anterior | 3–10 h |
| `asistencia` | % de clases asistidas | 50–100 % |
| `tareas_entregadas` | Proporción de tareas entregadas | 0–1 |
| `nota_parcial` | Nota del examen parcial | 0–10 |
| `aprueba` | ¿Aprobó el examen final? | 1 = sí / 0 = no |

**Regla generadora:** combinación ponderada de las 5 variables con ruido del 15 %.

In [2]:
N = 400

horas_estudio     = rng.uniform(0, 10, N)
horas_sueno       = rng.uniform(3, 10, N)
asistencia        = rng.uniform(50, 100, N)
tareas_entregadas = rng.uniform(0, 1, N)
nota_parcial      = rng.uniform(0, 10, N)

# Puntuación ponderada → aprueba si supera umbral
score = (
    0.30 * (horas_estudio / 10) +
    0.20 * ((horas_sueno - 3) / 7) +
    0.20 * ((asistencia - 50) / 50) +
    0.15 * tareas_entregadas +
    0.15 * (nota_parcial / 10)
)

aprueba_base = (score >= 0.55).astype(int)
ruido        = rng.random(N) < 0.15
aprueba      = np.where(ruido, 1 - aprueba_base, aprueba_base)

df = pd.DataFrame({
    "horas_estudio":     horas_estudio.round(1),
    "horas_sueno":       horas_sueno.round(1),
    "asistencia":        asistencia.round(1),
    "tareas_entregadas": tareas_entregadas.round(2),
    "nota_parcial":      nota_parcial.round(1),
    "aprueba":           aprueba,
})

FEATURES = ["horas_estudio", "horas_sueno", "asistencia", "tareas_entregadas", "nota_parcial"]
X = df[FEATURES].values
y = df["aprueba"].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Total estudiantes : {N}")
print(f"Aprueba (1)       : {aprueba.sum()} ({aprueba.mean():.0%})")
print(f"Reprueba (0)      : {(1-aprueba).sum()} ({1-aprueba.mean():.0%})")
print(f"\nTrain: {len(X_train)} filas | Test: {len(X_test)} filas")
df.head()

Total estudiantes : 400
Aprueba (1)       : 169 (42%)
Reprueba (0)      : 231 (58%)

Train: 320 filas | Test: 80 filas


,horas_estudio,horas_sueno,asistencia,tareas_entregadas,nota_parcial,aprueba
0,7.7,5.1,86.9,0.57,9.1,1
1,4.4,9.4,65.9,0.55,1.4,0
2,8.6,8.5,94.5,0.83,6.8,1
3,7.0,3.8,79.7,0.71,8.1,1
4,0.9,10.0,56.3,0.03,2.4,1
